In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import torch
import numpy as np
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm.notebook import tqdm

PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Import Dataset and all 3 Extractor Models
from src.data_pipeline import (
    CleanedGreekLettersDataset, 
    VGG16FeatureExtractor, 
    EarlyVGG16FeatureExtractor, 
    VGG16SPPFeatureExtractor
)

# Global Device Setup (MPS for Apple Silicon)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Define the new input directories generated by 01_preprocessing.ipynb
TRAIN_DIR = "../data/split_dataset/train"
TEST_DIR = "../data/split_dataset/test"

Using device: mps


In [2]:
def extract_and_save(model, transform, data_dir, output_dir, split_name, batch_size=128):
    """
    Handles data loading, GPU extraction, and saving for a specific split (train/test).
    """
    dataset = CleanedGreekLettersDataset(root_dir=data_dir, transform=transform)
    assert len(dataset) > 0, f"Critical Error: 0 images found in {data_dir}."
    
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    
    model = model.to(device)
    model.eval()
    
    all_features, all_labels = [], []
    
    print(f"Extracting {split_name} features from {len(dataset)} images...")
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc=f"{split_name.capitalize()} Set"):
            images = images.to(device)
            features = model(images)
            
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())
            
    X = np.vstack(all_features)
    y = np.concatenate(all_labels)
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Save with split_name to differentiate train vs test
    np.save(os.path.join(output_dir, f"X_{split_name}.npy"), X)
    np.save(os.path.join(output_dir, f"y_{split_name}.npy"), y)
    
    print(f"Saved {split_name} - X shape: {X.shape}, y shape: {y.shape}\n")

In [3]:
print("--- RUNNING EXTRACTION: LATE VGG-16 (BLOCK 5 + GAP) ---")

transform_224 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

model_late = VGG16FeatureExtractor()
out_dir_late = "../data/extracted_features"

extract_and_save(model_late, transform_224, TRAIN_DIR, out_dir_late, "train")
extract_and_save(model_late, transform_224, TEST_DIR, out_dir_late, "test")

# Free up memory before loading the next model
del model_late
torch.mps.empty_cache()

--- RUNNING EXTRACTION: LATE VGG-16 (BLOCK 5 + GAP) ---
Extracting train features from 100224 images...


Train Set:   0%|          | 0/783 [00:00<?, ?it/s]

Saved train - X shape: (100224, 512), y shape: (100224,)

Extracting test features from 10977 images...


Test Set:   0%|          | 0/86 [00:00<?, ?it/s]

Saved test - X shape: (10977, 512), y shape: (10977,)



In [4]:
print("--- RUNNING EXTRACTION: EARLY VGG-16 (BLOCK 3 + GAP) ---")

model_early = EarlyVGG16FeatureExtractor()
out_dir_early = "../data/extracted_features_early"

extract_and_save(model_early, transform_224, TRAIN_DIR, out_dir_early, "train")
extract_and_save(model_early, transform_224, TEST_DIR, out_dir_early, "test")

del model_early
torch.mps.empty_cache()

--- RUNNING EXTRACTION: EARLY VGG-16 (BLOCK 3 + GAP) ---
Extracting train features from 100224 images...


Train Set:   0%|          | 0/783 [00:00<?, ?it/s]

Saved train - X shape: (100224, 256), y shape: (100224,)

Extracting test features from 10977 images...


Test Set:   0%|          | 0/86 [00:00<?, ?it/s]

Saved test - X shape: (10977, 256), y shape: (10977,)



In [5]:
print("--- RUNNING EXTRACTION: SPP VGG-16 (BLOCK 5 + SPP 2x2) ---")

# Must use 256x256 to ensure perfect divisibility for MPS hardware pooling
transform_256 = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

model_spp = VGG16SPPFeatureExtractor()
out_dir_spp = "../data/extracted_features_SPP"

extract_and_save(model_spp, transform_256, TRAIN_DIR, out_dir_spp, "train")
extract_and_save(model_spp, transform_256, TEST_DIR, out_dir_spp, "test")

del model_spp
torch.mps.empty_cache()

--- RUNNING EXTRACTION: SPP VGG-16 (BLOCK 5 + SPP 2x2) ---
Extracting train features from 100224 images...


Train Set:   0%|          | 0/783 [00:00<?, ?it/s]

Saved train - X shape: (100224, 2048), y shape: (100224,)

Extracting test features from 10977 images...


Test Set:   0%|          | 0/86 [00:00<?, ?it/s]

Saved test - X shape: (10977, 2048), y shape: (10977,)



# Note!
**the following cells are now redundant and have thus been marked as raw text so they are not accidentally executed.**

Maintained for archival purposes.